# Lunar Diviner temperature preprocessing

Ported from `notebooks/preprocessing/06_diviner_temperature.ipynb`. Original source files are unchanged. Inputs and all generated data stay under `darrel stuff/data/`.

Requires all 144 seasonal table/XML pairs and the same seven SDEM footprints. The original XML-driven chunk parser, lunar projection, special-value handling and per-region CSV schema are retained. Full processing scans about 30.74 GB; no scan is started when opening this notebook. FILE_LIMIT=1 creates an explicitly labelled pilot, not a full input for notebook 25.

**Code copied first; not executed locally.** No historical outputs are presented as new results. Install the original processing dependencies and read `../../EDA_Notes.md` before Run All.


In [4]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "outline.md").is_file() and (p / "notebooks").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository or one of its subfolders")
DATA_ROOT = ROOT / "darrel stuff/data"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "darrel stuff/EDA"))
from environment_workflow import LocalRun as Run, latest_completed_run, select_climate_manifest

print("Source data:", DATA_ROOT)
print("Processed outputs:", DATA_ROOT / "processed")


Source data: c:\Darrel\Self-Projects\1 group proj\NASA-Space-Apps-Challenge-2026\darrel stuff\data
Processed outputs: c:\Darrel\Self-Projects\1 group proj\NASA-Space-Apps-Challenge-2026\darrel stuff\data\processed


In [5]:
import json
import re
import numpy as np
import pandas as pd
import rasterio
from rasterio.crs import CRS
from pyproj import Transformer
from notebooks.shared.processing import MOON_CRS, MOON_GEOGRAPHIC, diviner_chunks

FILE_LIMIT = None  # Set 1 for a pilot; None selects all 144 table/label pairs.
CHUNK_ROWS = 100_000
EXPORT_INVALID_TEMPERATURE_CELLS = True  # Preserve missingness within the selected footprint.
base = DATA_ROOT / "moon/diviner/seasonal"
labels = sorted(base.glob("pcp_avg_tbol_pols_ls*_slon*_240.xml"))
assert len(labels) == 144
if FILE_LIMIT is not None:
    assert isinstance(FILE_LIMIT, int) and FILE_LIMIT > 0
    labels = labels[:FILE_LIMIT]
regions = {}
region_sources = {}
for directory in sorted((DATA_ROOT / "moon/sdem/regions").iterdir()):
    if not directory.is_dir():
        continue
    paths = list(directory.glob("*_GLDELEV_005.tif"))
    assert len(paths) == 1
    with rasterio.open(paths[0]) as src:
        assert src.crs == CRS.from_string(MOON_CRS)
        regions[directory.name] = tuple(src.bounds)
        region_sources[directory.name] = paths[0]
assert len(regions) == 7
project = Transformer.from_crs(MOON_GEOGRAPHIC, MOON_CRS, always_xy=True)
print("Selected files:", len(labels), "regions:", len(regions))


Selected files: 144 regions: 7


In [6]:
config = {"file_limit": FILE_LIMIT, "chunk_rows": CHUNK_ROWS, "regions": regions,
          "scope": "full seasonal collection" if len(labels) == 144 else "pilot subset",
          "native_grid_m": 240, "temperature": "average bolometric brightness temperature, K",
          "export_invalid_temperature_cells": EXPORT_INVALID_TEMPERATURE_CELLS}
with Run(ROOT, "diviner_temperature", config, "Moon/Thermal/06_diviner_temperature.ipynb") as run:
    run.input(DATA_ROOT / "moon/diviner/download_manifest.csv")
    for source in region_sources.values():
        run.input(source)
    summaries = []
    for label in labels:
        run.input(label)
        run.input(label.with_suffix(".tab"))
        match = re.search(r"ls(\d+)_slon(\d+)", label.stem)
        assert match is not None
        seasonal_bin, subsolar_bin = int(match[1]), int(match[2])
        selected = {name: [] for name in regions}
        for chunk in diviner_chunks(label, CHUNK_ROWS):
            coordinate_valid = np.isfinite(chunk["clon"]) & np.isfinite(chunk["clat"])
            coordinate_valid &= chunk["clon"].between(-180, 180) & chunk["clat"].between(-90, -79)
            chunk = chunk.loc[coordinate_valid].copy()
            x, y = project.transform(chunk["clon"].to_numpy(), chunk["clat"].to_numpy())
            chunk["x_m"], chunk["y_m"] = x, y
            chunk["valid_temperature"] = np.isfinite(chunk["tbol"]) & (chunk["tbol"] > 0)
            chunk.loc[~chunk["valid_temperature"], "tbol"] = np.nan
            for name, (west, south, east, north) in regions.items():
                inside = (x >= west) & (x < east) & (y >= south) & (y < north)
                if not EXPORT_INVALID_TEMPERATURE_CELLS:
                    inside &= chunk["valid_temperature"].to_numpy()
                if inside.any():
                    selected[name].append(chunk.loc[inside, ["clon", "clat", "x_m", "y_m", "ltim", "tbol", "valid_temperature"]].copy())
        for name, parts in selected.items():
            columns = ["clon", "clat", "x_m", "y_m", "ltim", "tbol", "valid_temperature"]
            frame = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=pd.Index(columns))
            if frame.duplicated(["clon", "clat"]).any():
                raise ValueError(f"Duplicate native cell centers in {label.name}, {name}")
            frame["seasonal_ls_bin"] = seasonal_bin
            frame["subsolar_longitude_bin"] = subsolar_bin
            frame["source_product"] = label.stem
            values = frame["tbol"].dropna().astype(float)
            summaries.append({"region": name, "source_product": label.stem, "seasonal_ls_bin": seasonal_bin,
                "subsolar_longitude_bin": subsolar_bin, "selected_rows": len(frame), "valid_temperature_rows": len(values),
                "missing_temperature_rows": len(frame) - len(values), "status": "has_data" if len(values) else "no_valid_cells",
                "spatial_mean_of_bin_average_K": float(values.mean()) if len(values) else None,
                "spatial_min_of_bin_average_K": float(values.min()) if len(values) else None,
                "spatial_max_of_bin_average_K": float(values.max()) if len(values) else None})
            frame.to_csv(run.path(f"moon/{name}/native_240m/{label.stem}.csv.gz"), index=False,
                         compression={"method": "gzip", "mtime": 0})
        print(label.name, "selected")
    summary_frame = pd.DataFrame(summaries)
    summary_frame.to_csv(run.path("regional_bin_summary.csv"), index=False)
    assert len(summary_frame) == len(labels) * len(regions)
    run.record["empty_region_bins"] = int((summary_frame["valid_temperature_rows"] == 0).sum())
    run.record["summary_rows"] = len(summary_frame)
    run.path("interpretation.json").write_text(json.dumps({"missing_bins": "empty, not zero",
        "source_time_window": "2009-07-05 to 2020-03-15 per product labels; cumulative seasonal bins, not a monthly time series",
        "pixel_support": "cell centers inside each SDEM bounding rectangle; not image-support masking",
        "temperature_statistics": "spatial statistics of average brightness temperature within each source bin"}, indent=2))
print("Completed:", run.directory)
print(summary_frame.groupby("region")["valid_temperature_rows"].agg(["min", "max"]))


pcp_avg_tbol_pols_ls030_slon01_240.xml selected
pcp_avg_tbol_pols_ls030_slon02_240.xml selected
pcp_avg_tbol_pols_ls030_slon03_240.xml selected
pcp_avg_tbol_pols_ls030_slon04_240.xml selected
pcp_avg_tbol_pols_ls030_slon05_240.xml selected
pcp_avg_tbol_pols_ls030_slon06_240.xml selected
pcp_avg_tbol_pols_ls030_slon07_240.xml selected
pcp_avg_tbol_pols_ls030_slon08_240.xml selected
pcp_avg_tbol_pols_ls030_slon09_240.xml selected
pcp_avg_tbol_pols_ls030_slon10_240.xml selected
pcp_avg_tbol_pols_ls030_slon11_240.xml selected
pcp_avg_tbol_pols_ls030_slon12_240.xml selected
pcp_avg_tbol_pols_ls030_slon13_240.xml selected
pcp_avg_tbol_pols_ls030_slon14_240.xml selected
pcp_avg_tbol_pols_ls030_slon15_240.xml selected
pcp_avg_tbol_pols_ls030_slon16_240.xml selected
pcp_avg_tbol_pols_ls030_slon17_240.xml selected
pcp_avg_tbol_pols_ls030_slon18_240.xml selected
pcp_avg_tbol_pols_ls030_slon19_240.xml selected
pcp_avg_tbol_pols_ls030_slon20_240.xml selected
pcp_avg_tbol_pols_ls030_slon21_240.xml s

## Interpretation

These are cumulative phase-bin averages of bolometric brightness temperature, not dated mission temperatures or extremes. Missing cells/bins remain missing. The denominator is native cell centers inside each original SDEM rectangle, not its image-support mask.